In [1]:
import json
import hashlib
import sys
from urllib.request import Request, build_opener, ProxyHandler

MODELO = "deepseek-r1:8b-0528-qwen3-q4_K_M"
API_LOCAL = "http://127.0.0.1:11434"

# Archivos comunes de comparación
with open("../casos_comparacion_v1.json", "rb") as f:
    bytes_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    bytes_config = f.read()

casos = json.loads(bytes_casos)
configuracion = json.loads(bytes_config)

sha_casos = hashlib.sha256(bytes_casos).hexdigest()
sha_config = hashlib.sha256(bytes_config).hexdigest()

# Verificar integridad de los archivos
assert sha_casos == configuracion["cases_sha256"]
assert sha_config == (
    "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"
)

assert [c["case_id"] for c in casos] == [
    "S-005", "S-006", "S-007",
    "S-008", "S-009", "S-010"
]

assert configuracion["prompt_version"] == "V3"
assert configuracion["api_url"] == API_LOCAL + "/api/chat"

# Conexión local
cliente_local = build_opener(ProxyHandler({}))

with cliente_local.open(API_LOCAL + "/api/version", timeout=10) as f:
    version_ollama = json.load(f)["version"]

with cliente_local.open(API_LOCAL + "/api/tags", timeout=10) as f:
    modelos = json.load(f)["models"]

modelo_info = next(
    (m for m in modelos if m["name"] == MODELO),
    None
)

assert modelo_info is not None, "Modelo no instalado."

# Consultar capacidades y parámetros
peticion = Request(
    API_LOCAL + "/api/show",
    data=json.dumps({"model": MODELO}).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST"
)

with cliente_local.open(peticion, timeout=15) as f:
    detalles = json.load(f)

print("Python:", sys.version.split()[0])
print("Ollama:", version_ollama)
print("Modelo:", MODELO)
print("Identificador:", modelo_info["digest"][:12])
print("Casos verificados:", len(casos))
print("SHA256:", sha_casos)
print("Prompt:", configuracion["prompt_version"])
print("Capacidades:", detalles.get("capabilities"))
print("Thinking:", detalles.get("thinking"))
print("Parámetros:", detalles.get("parameters"))

print("\nArchivos y conexión: correctos")


Python: 3.11.16
Ollama: 0.40.13
Modelo: deepseek-r1:8b-0528-qwen3-q4_K_M
Identificador: 6995872bfe4c
Casos verificados: 6
SHA256: f73d159bb43cc9c93d2768e997052f4381a2c24bb046c82826b1793c1b8da37d
Prompt: V3
Capacidades: ['tools', 'thinking', 'completion']
Thinking: None
Parámetros: temperature                    0.6
top_p                          0.95
stop                           "<｜begin▁of▁sentence｜>"
stop                           "<｜end▁of▁sentence｜>"
stop                           "<｜User｜>"
stop                           "<｜Assistant｜>"

Archivos y conexión: correctos


In [2]:
import json
import copy
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request

ARCHIVO = "resultados.jsonl"
CASO_ID = "DIAG-DSR1-8B-01"

# 1. Mismo caso diagnóstico de los modelos anteriores
contenido = (
    "Autores: Autor X y Autor Y.\n"
    "Predicción recibida: favorable (simulada).\n"
    "Modelo predictor y puntuación: no disponibles.\n"
    "[E1] Autor X tiene un artículo sobre bases de datos académicas.\n"
    "[E2] Autor Y tiene un artículo sobre recuperación de información.\n"
    "No se proporcionan datos de colaboración previa."
)

# 2. Recuperar configuración compartida V3
solicitud = copy.deepcopy(
    configuracion["request_defaults"]
)

solicitud["model"] = MODELO
solicitud["think"] = True

# Parámetros del diagnóstico
solicitud["options"].update({
    "num_ctx": 8192,
    "num_predict": 6144,
    "temperature": 0.6,
    "top_p": 0.95
})

# Conservar el prompt V3 original
solicitud["messages"] = [
    {
        "role": "system",
        "content": configuracion["system_prompt"]
    },
    {
        "role": "user",
        "content": configuracion["user_prompt_template"].format(
            case_id=CASO_ID,
            contenido=contenido
        )
    }
]

assert "thinking" in detalles.get("capabilities", [])
assert solicitud["stream"] is False

# 3. Evitar duplicar el diagnóstico
try:
    with open(ARCHIVO, encoding="utf-8") as f:
        anteriores = [
            json.loads(line)
            for line in f if line.strip()
        ]
except FileNotFoundError:
    anteriores = []

assert not any(
    r.get("case_id") == CASO_ID
    for r in anteriores
), "El diagnóstico ya está registrado."

# 4. Preparar registro
registro = {
    "execution_id": str(uuid4()),
    "case_id": CASO_ID,
    "grupo": "diagnostico",
    "configuracion_modelo": "deepseek_r1_8b_thinking_v1",
    "modelo_digest": modelo_info["digest"],
    "ollama_version": version_ollama,
    "sha256_casos": sha_casos,
    "sha256_configuracion": sha_config,
    "fecha_inicio_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "solicitud": solicitud,
    "reintentos": 0
}

inicio = time.perf_counter()

# 5. Ejecutar en Ollama local
try:
    peticion = Request(
        configuracion["api_url"],
        data=json.dumps(
            solicitud, ensure_ascii=False
        ).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    with cliente_local.open(peticion, timeout=360) as conexion:
        respuesta = json.load(conexion)

    registro["respuesta_original"] = respuesta

    mensaje = respuesta.get("message", {})

    if respuesta.get("done_reason") == "length":
        registro["estado"] = "truncada"

    elif (
        respuesta.get("done_reason") == "stop"
        and mensaje.get("content", "").strip()
        and mensaje.get("thinking")
    ):
        registro["estado"] = "ok"

    else:
        registro["estado"] = "revisar"

except Exception as error:
    registro["estado"] = "error"
    registro["error_tipo"] = type(error).__name__
    registro["error"] = str(error)

registro["tiempo_cliente_s"] = round(
    time.perf_counter() - inicio, 3
)

registro["fecha_fin_utc"] = datetime.now(
    timezone.utc
).isoformat()

# 6. Guardar solicitud y respuesta completas
with open(ARCHIVO, "a", encoding="utf-8") as f:
    f.write(
        json.dumps(registro, ensure_ascii=False) + "\n"
    )

# 7. Mostrar métricas y respuesta final
print("Estado:", registro["estado"])
print("Tiempo cliente:", registro["tiempo_cliente_s"], "s")

if "respuesta_original" in registro:
    r = registro["respuesta_original"]
    mensaje = r.get("message", {})

    print("Tokens entrada:", r.get("prompt_eval_count"))
    print("Tokens generados:", r.get("eval_count"))
    print("Terminación:", r.get("done_reason"))
    print("Razonamiento separado:", bool(
        mensaje.get("thinking")
    ))

    for campo in (
        "total_duration",
        "load_duration",
        "prompt_eval_duration",
        "eval_duration"
    ):
        valor = r.get(campo)
        if valor is not None:
            print(campo, round(valor / 1e9, 3), "s")

    print("\nRESPUESTA FINAL:\n")
    print(mensaje.get("content", ""))

else:
    print("ERROR:", registro.get("error"))

print("\nRegistro guardado en resultados.jsonl")


Estado: ok
Tiempo cliente: 32.583 s
Tokens entrada: 414
Tokens generados: 1068
Terminación: stop
Razonamiento separado: True
total_duration 32.581 s
load_duration 4.398 s
prompt_eval_duration 0.682 s
eval_duration 27.494 s

RESPUESTA FINAL:


**Evidencia:**
La evidencia proporcionada consta de dos elementos: [E1] indica que el autor X tiene un artículo relacionado con bases de datos académicas, y [E2] muestra que el autor Y tiene un artículo relacionado con la recuperación de información. Ambos temas pertenecen al dominio de la informática y la gestión de la información, específicamente en torno a los sistemas de información. Es posible que estos temas estén relacionados, ya que la recuperación de información es una aplicación clave de las bases de datos.

**Interpretación:**
La interpretación de la evidencia sugiere una posible similitud semántica entre los trabajos de los autores X e Y. Aunque no se documenta una colaboración previa directa, la temática cercana de sus publicaciones p

In [3]:
import json
import copy
import hashlib
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request, build_opener, ProxyHandler

MODELO = "deepseek-r1:8b-0528-qwen3-q4_K_M"
API = "http://127.0.0.1:11434"
ARCHIVO = "resultados.jsonl"

# 1. Verificar archivos compartidos
with open("../casos_comparacion_v1.json", "rb") as f:
    bytes_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    bytes_config = f.read()

casos = json.loads(bytes_casos)
config = json.loads(bytes_config)

sha_casos = hashlib.sha256(bytes_casos).hexdigest()
sha_config = hashlib.sha256(bytes_config).hexdigest()

assert sha_casos == config["cases_sha256"]
assert sha_config == (
    "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"
)

assert [c["case_id"] for c in casos] == [
    "S-005", "S-006", "S-007",
    "S-008", "S-009", "S-010"
]

assert config["prompt_version"] == "V3"
assert config["api_url"] == API + "/api/chat"

# 2. Recuperar el diagnóstico exitoso
with open(ARCHIVO, encoding="utf-8") as f:
    anteriores = [
        json.loads(line) for line in f if line.strip()
    ]

diagnostico = next(
    r for r in anteriores
    if r["case_id"] == "DIAG-DSR1-8B-01"
)

assert diagnostico["estado"] == "ok"
assert diagnostico["solicitud"]["model"] == MODELO
assert diagnostico["solicitud"]["think"] is True

opciones = diagnostico["solicitud"]["options"]

assert opciones["num_ctx"] == 8192
assert opciones["num_predict"] == 6144
assert opciones["temperature"] == 0.6
assert opciones["top_p"] == 0.95

assert diagnostico["solicitud"]["messages"][0]["content"] == (
    config["system_prompt"]
)

# 3. Verificar identidad del modelo local
cliente = build_opener(ProxyHandler({}))

with cliente.open(API + "/api/tags", timeout=10) as f:
    modelos = json.load(f)["models"]

modelo_info = next(
    m for m in modelos if m["name"] == MODELO
)

assert modelo_info["digest"] == diagnostico["modelo_digest"]

# Evitar duplicar ejecuciones
registrados = {
    r["case_id"] for r in anteriores
    if r.get("configuracion_modelo") == "deepseek_r1_8b_thinking_v1"
    and r.get("grupo") == "comparacion"
}

# 4. Ejecutar los seis casos
with open(ARCHIVO, "a", encoding="utf-8") as archivo:

    for caso in casos:

        case_id = caso["case_id"]

        if case_id in registrados:
            print(case_id, "| ya registrado")
            continue

        # Reutilizar la configuración del diagnóstico
        solicitud = copy.deepcopy(
            diagnostico["solicitud"]
        )

        # Cambiar únicamente el caso y su evidencia
        solicitud["messages"][1]["content"] = (
            config["user_prompt_template"].format(
                case_id=case_id,
                contenido=caso["contenido"]
            )
        )

        registro = {
            "execution_id": str(uuid4()),
            "case_id": case_id,
            "tipo": caso["tipo"],
            "grupo": "comparacion",
            "configuracion_modelo": "deepseek_r1_8b_thinking_v1",
            "prompt_version": "V3",
            "modelo_digest": modelo_info["digest"],
            "ollama_version": diagnostico["ollama_version"],
            "sha256_casos": sha_casos,
            "sha256_configuracion": sha_config,
            "fecha_inicio_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "solicitud": solicitud,
            "reintentos": 0
        }

        inicio = time.perf_counter()

        try:
            peticion = Request(
                config["api_url"],
                data=json.dumps(
                    solicitud, ensure_ascii=False
                ).encode("utf-8"),
                headers={"Content-Type": "application/json"},
                method="POST"
            )

            with cliente.open(peticion, timeout=360) as conexion:
                respuesta = json.load(conexion)

            registro["respuesta_original"] = respuesta

            mensaje = respuesta.get("message", {})

            if respuesta.get("done_reason") == "length":
                registro["estado"] = "truncada"

            elif (
                respuesta.get("done_reason") == "stop"
                and mensaje.get("content", "").strip()
                and mensaje.get("thinking")
            ):
                registro["estado"] = "ok"

            else:
                registro["estado"] = "revisar"

        except Exception as error:
            registro["estado"] = "error"
            registro["error_tipo"] = type(error).__name__
            registro["error"] = str(error)

        registro["tiempo_cliente_s"] = round(
            time.perf_counter() - inicio, 3
        )

        registro["fecha_fin_utc"] = datetime.now(
            timezone.utc
        ).isoformat()

        # Guardar inmediatamente cada resultado
        archivo.write(
            json.dumps(registro, ensure_ascii=False) + "\n"
        )
        archivo.flush()

        r = registro.get("respuesta_original", {})

        print(
            case_id,
            "|", registro["estado"],
            "|", r.get("eval_count", "-"), "tokens",
            "|", registro["tiempo_cliente_s"], "s"
        )

print("\nArchivo actualizado:", ARCHIVO)


S-005 | ok | 934 tokens | 17.925 s
S-006 | ok | 701 tokens | 13.742 s
S-007 | ok | 1327 tokens | 24.843 s
S-008 | ok | 1123 tokens | 21.707 s
S-009 | ok | 1497 tokens | 25.565 s
S-010 | ok | 1290 tokens | 22.791 s

Archivo actualizado: resultados.jsonl


In [4]:
import json
import copy
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request, build_opener, ProxyHandler

ARCHIVO = "resultados.jsonl"
MODELO = "deepseek-r1:8b-0528-qwen3-q4_K_M"
API = "http://127.0.0.1:11434"
CASO_ID = "DIAG-DSR1-8B-02"

# 1. Recuperar el diagnóstico con razonamiento
with open(ARCHIVO, encoding="utf-8") as f:
    registros = [
        json.loads(line) for line in f if line.strip()
    ]

assert not any(
    r.get("case_id") == CASO_ID
    for r in registros
), "El diagnóstico ya existe."

anterior = next(
    r for r in registros
    if r["case_id"] == "DIAG-DSR1-8B-01"
)

assert anterior["estado"] == "ok"
assert anterior["solicitud"]["model"] == MODELO
assert anterior["solicitud"]["think"] is True

# 2. Conservar el mismo prompt y parámetros
solicitud = copy.deepcopy(anterior["solicitud"])

# Único cambio experimental
solicitud["think"] = False

assert solicitud["options"]["num_ctx"] == 8192
assert solicitud["options"]["num_predict"] == 6144
assert solicitud["options"]["temperature"] == 0.6
assert solicitud["options"]["top_p"] == 0.95

# 3. Verificar identidad del modelo local
cliente = build_opener(ProxyHandler({}))

with cliente.open(API + "/api/tags", timeout=10) as f:
    instalados = json.load(f)["models"]

modelo_actual = next(
    m for m in instalados if m["name"] == MODELO
)

assert modelo_actual["digest"] == anterior["modelo_digest"]

# 4. Preparar nuevo registro
registro = {
    "execution_id": str(uuid4()),
    "case_id": CASO_ID,
    "grupo": "diagnostico",
    "configuracion_modelo": "deepseek_r1_8b_directa_v1",
    "referencia_execution_id": anterior["execution_id"],
    "modelo_digest": modelo_actual["digest"],
    "ollama_version": anterior["ollama_version"],
    "sha256_casos": anterior["sha256_casos"],
    "sha256_configuracion": anterior["sha256_configuracion"],
    "fecha_inicio_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "solicitud": solicitud,
    "reintentos": 0
}

inicio = time.perf_counter()

# 5. Ejecutar el diagnóstico directo
try:
    peticion = Request(
        API + "/api/chat",
        data=json.dumps(
            solicitud, ensure_ascii=False
        ).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    with cliente.open(peticion, timeout=360) as conexion:
        respuesta = json.load(conexion)

    registro["respuesta_original"] = respuesta

    mensaje = respuesta.get("message", {})
    contenido = mensaje.get("content", "")

    if respuesta.get("done_reason") == "length":
        registro["estado"] = "truncada"

    elif (
        mensaje.get("thinking")
        or "<think>" in contenido
        or "</think>" in contenido
    ):
        registro["estado"] = "revisar_modo"

    elif (
        respuesta.get("done_reason") == "stop"
        and contenido.strip()
    ):
        registro["estado"] = "ok"

    else:
        registro["estado"] = "revisar"

except Exception as error:
    registro["estado"] = "error"
    registro["error_tipo"] = type(error).__name__
    registro["error"] = str(error)

registro["tiempo_cliente_s"] = round(
    time.perf_counter() - inicio, 3
)

registro["fecha_fin_utc"] = datetime.now(
    timezone.utc
).isoformat()

# 6. Conservar ambas ejecuciones
with open(ARCHIVO, "a", encoding="utf-8") as f:
    f.write(
        json.dumps(registro, ensure_ascii=False) + "\n"
    )

# 7. Mostrar resultados
print("Caso:", CASO_ID)
print("Estado:", registro["estado"])
print("Tiempo cliente:", registro["tiempo_cliente_s"], "s")

if "respuesta_original" in registro:
    r = registro["respuesta_original"]
    mensaje = r.get("message", {})

    print("Tokens entrada:", r.get("prompt_eval_count"))
    print("Tokens generados:", r.get("eval_count"))
    print("Terminación:", r.get("done_reason"))
    print("Razonamiento separado:", bool(
        mensaje.get("thinking")
    ))

    print("\nRESPUESTA FINAL:\n")
    print(mensaje.get("content", ""))

else:
    print("ERROR:", registro.get("error"))

print("\nRegistro añadido a resultados.jsonl")


Caso: DIAG-DSR1-8B-02
Estado: revisar_modo
Tiempo cliente: 33.554 s
Tokens entrada: 414
Tokens generados: 1884
Terminación: stop
Razonamiento separado: True

RESPUESTA FINAL:


### Evidencia
La evidencia proporcionada consta de dos elementos: [E1] Autor X tiene un artículo sobre bases de datos académicas, y [E2] Autor Y tiene un artículo sobre recuperación de información. Estos datos indican que ambos autores han publicado en áreas temáticas relacionadas, pero no mencionan ninguna colaboración previa o detalles específicos de sus trabajos.

### Interpretación
La evidencia [E1] y [E2] muestra una similitud semántica entre los temas de los artículos, ya que la recuperación de información es una subárea directamente relacionada con las bases de datos académicas. Esto sugiere que los autores comparten competencias en un campo disciplinario afín, lo que podría hacer sus investigaciones complementarias y relevantes para futuras colaboraciones en temas similares. La predicción favorable de co